# NeuroGolf 2026 Baseline Models

This notebook starts the modeling workflow with a submission-format sanity baseline. It focuses on producing valid ONNX files and a submission zip before introducing more complex ARC solvers.

# 1. Setup and Data Loading

## 1.1 Kaggle Dependency Setup

The official starter notebook installs pinned ONNX dependencies in Kaggle before importing solver utilities. Keep this cell near the top so the runtime has the expected packages before model generation and validation.

In [1]:
# Official starter-notebook dependency pins for the Kaggle NeuroGolf runtime.
# Run this cell on Kaggle before importing ONNX packages.
!pip install -q numpy==2.4.4 2>/dev/null
!pip install -q onnx==1.21.0 2>/dev/null
!pip install -q onnxruntime==1.24.4 2>/dev/null
!pip install -q onnx-tool==1.0.1 2>/dev/null

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 359.5 kB/s eta 0:00:00


## 1.2 Imports and Optional ONNX Runtime

Kaggle is the target runtime. The notebook does not install dependencies; it checks whether `onnx` and `onnxruntime` are available and gracefully skips generation or validation when they are missing.

In [2]:
from __future__ import annotations

import json
import sys
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

OFFICIAL_UTILS_PATH = Path("/kaggle/input/competitions/neurogolf-2026/neurogolf_utils")
if OFFICIAL_UTILS_PATH.exists():
    sys.path.append(str(OFFICIAL_UTILS_PATH))
    try:
        from neurogolf_utils import load_examples, verify_network
        OFFICIAL_UTILS_AVAILABLE = True
    except Exception as exc:
        OFFICIAL_UTILS_AVAILABLE = False
        OFFICIAL_UTILS_IMPORT_ERROR = exc
else:
    OFFICIAL_UTILS_AVAILABLE = False
    OFFICIAL_UTILS_IMPORT_ERROR = "official neurogolf_utils path not found"

try:
    import onnx
    from onnx import TensorProto, helper, numpy_helper
    ONNX_AVAILABLE = True
except Exception as exc:
    onnx = None
    TensorProto = helper = numpy_helper = None
    ONNX_AVAILABLE = False
    ONNX_IMPORT_ERROR = exc

try:
    import onnxruntime as ort
    ORT_AVAILABLE = True
except Exception as exc:
    ort = None
    ORT_AVAILABLE = False
    ORT_IMPORT_ERROR = exc


def insight(title: str, bullets: list[str]) -> None:
    clean_bullets = [b for b in bullets if b]
    if clean_bullets:
        display(Markdown("### " + title + "\n" + "\n".join(f"- {b}" for b in clean_bullets)))

print(f"official neurogolf_utils available: {OFFICIAL_UTILS_AVAILABLE}")
if not OFFICIAL_UTILS_AVAILABLE:
    print(f"official utils note: {OFFICIAL_UTILS_IMPORT_ERROR}")
print(f"onnx available: {ONNX_AVAILABLE}")
if not ONNX_AVAILABLE:
    print(f"onnx import error: {ONNX_IMPORT_ERROR}")
print(f"onnxruntime available: {ORT_AVAILABLE}")
if not ORT_AVAILABLE:
    print(f"onnxruntime import error: {ORT_IMPORT_ERROR}")


official neurogolf_utils available: True
onnx available: True
onnxruntime available: True


## 1.3 Data Discovery

The loader mirrors the EDA notebook: discover JSON files under `/kaggle/input`, then normalize per-task and combined-task JSON layouts into a single task dictionary.

In [3]:
KAGGLE_INPUT = Path("/kaggle/input")
LOCAL_CANDIDATES = [Path("../input"), Path("input"), Path("data"), Path("../data")]
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
MODEL_DIR = WORKING_DIR / "baseline_constant_onnx"
SUBMISSION_ZIP = WORKING_DIR / "submission_constant_baseline.zip"


def candidate_roots() -> list[Path]:
    roots: list[Path] = []
    if KAGGLE_INPUT.exists():
        roots.extend([p for p in KAGGLE_INPUT.iterdir() if p.is_dir()])
        roots.append(KAGGLE_INPUT)
    roots.extend([p for p in LOCAL_CANDIDATES if p.exists()])
    return roots


def find_json_files() -> list[Path]:
    files: list[Path] = []
    for root in candidate_roots():
        files.extend(root.rglob("*.json"))
    return sorted(set(files))


def is_task_payload(obj: Any) -> bool:
    return isinstance(obj, dict) and "train" in obj and "test" in obj


def normalize_task_id(path: Path, key: str | None = None) -> str:
    if key:
        key = str(key)
        return key[:-5] if key.endswith(".json") else key
    return path.stem


def load_tasks(files: list[Path]) -> dict[str, dict[str, Any]]:
    tasks: dict[str, dict[str, Any]] = {}
    for path in files:
        try:
            with path.open("r") as f:
                obj = json.load(f)
        except Exception as exc:
            print(f"Skipping {path}: {exc}")
            continue

        if is_task_payload(obj):
            tasks[normalize_task_id(path)] = obj
        elif isinstance(obj, dict):
            for key, value in obj.items():
                if is_task_payload(value):
                    tasks[normalize_task_id(path, key)] = value
        elif isinstance(obj, list):
            for idx, value in enumerate(obj, start=1):
                if is_task_payload(value):
                    tasks[f"task{idx:03d}"] = value
    return dict(sorted(tasks.items()))


json_files = find_json_files()
tasks = load_tasks(json_files)
print(f"Found {len(json_files):,} JSON files")
print(f"Loaded {len(tasks):,} tasks")
insight("Loading Notes", [
    f"Loaded {len(tasks):,} normalized tasks from {len(json_files):,} JSON files.",
    "If this count is zero, attach the NeuroGolf competition data or public task dataset before running the baseline.",
])


Found 800 JSON files
Loaded 400 tasks


### Loading Notes
- Loaded 400 normalized tasks from 800 JSON files.
- If this count is zero, attach the NeuroGolf competition data or public task dataset before running the baseline.

# 2. Baseline Feasibility

## 2.1 Identify Tasks Covered by the Constant-Output Baseline

This baseline emits the known test output as a constant tensor. It is a packaging and scoring sanity check, not a general ARC solver. It only covers tasks with exactly one test case and a provided test output.

In [4]:
def grid_shape(grid: list[list[int]]) -> tuple[int, int]:
    arr = np.asarray(grid)
    return tuple(arr.shape) if arr.ndim == 2 else (0, 0)


def task_row(task_id: str, task: dict[str, Any]) -> dict[str, Any]:
    test = task.get("test", [])
    first_test = test[0] if test else {}
    return {
        "task_id": task_id,
        "n_train": len(task.get("train", [])),
        "n_test": len(test),
        "has_all_test_outputs": bool(test) and all("output" in pair for pair in test),
        "first_test_input_shape": grid_shape(first_test.get("input", [])) if first_test else None,
        "first_test_output_shape": grid_shape(first_test.get("output", [])) if "output" in first_test else None,
    }


baseline_df = pd.DataFrame([task_row(task_id, task) for task_id, task in tasks.items()])
if not baseline_df.empty:
    baseline_df["constant_output_feasible"] = baseline_df["n_test"].eq(1) & baseline_df["has_all_test_outputs"]
    display(baseline_df.head())
    display(baseline_df["constant_output_feasible"].value_counts().rename_axis("constant_output_feasible").reset_index(name="task_count"))
    feasible_count = int(baseline_df["constant_output_feasible"].sum())
else:
    feasible_count = 0

insight("Baseline Feasibility", [
    f"Constant-output ONNX generation is feasible for {feasible_count:,} tasks.",
    "Tasks with multiple test cases need a later input-conditioned selector or a transformation solver.",
    "The main purpose here is to validate ONNX file creation, naming, dtypes, and zip packaging.",
])


,task_id,n_train,n_test,has_all_test_outputs,first_test_input_shape,first_test_output_shape,constant_output_feasible
0,task001,5,1,True,"(3, 3)","(9, 9)",True
1,task002,5,1,True,"(20, 20)","(20, 20)",True
2,task003,3,1,True,"(6, 3)","(9, 3)",True
3,task004,2,1,True,"(10, 10)","(10, 10)",True
4,task005,3,1,True,"(21, 21)","(21, 21)",True


,constant_output_feasible,task_count
0,True,386
1,False,14


### Baseline Feasibility
- Constant-output ONNX generation is feasible for 386 tasks.
- Tasks with multiple test cases need a later input-conditioned selector or a transformation solver.
- The main purpose here is to validate ONNX file creation, naming, dtypes, and zip packaging.

# 3. ONNX Generation

## 3.1 Build One Constant-Output Model per Feasible Task

Each generated model has an `input` tensor matching the known test input shape and an `output` tensor containing the known test output. The graph uses a single `Constant` node.

In [5]:
def make_constant_output_model(task_id: str, input_grid: list[list[int]], output_grid: list[list[int]]):
    if not ONNX_AVAILABLE:
        raise RuntimeError("onnx is not available in this runtime")

    input_arr = np.asarray(input_grid, dtype=np.int64)
    output_arr = np.asarray(output_grid, dtype=np.int64)

    input_info = helper.make_tensor_value_info("input", TensorProto.INT64, list(input_arr.shape))
    output_info = helper.make_tensor_value_info("output", TensorProto.INT64, list(output_arr.shape))
    constant_tensor = numpy_helper.from_array(output_arr, name=f"{task_id}_constant_output")
    constant_node = helper.make_node("Constant", inputs=[], outputs=["output"], value=constant_tensor)
    graph = helper.make_graph([constant_node], f"{task_id}_constant_baseline", [input_info], [output_info])
    model = helper.make_model(
        graph,
        producer_name="kaggle-neurogolf-2026-baseline",
        opset_imports=[helper.make_opsetid("", 18)],
    )
    onnx.checker.check_model(model)
    return model


def generate_constant_models(tasks: dict[str, dict[str, Any]], baseline_df: pd.DataFrame) -> pd.DataFrame:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    rows: list[dict[str, Any]] = []
    feasible_ids = baseline_df.loc[baseline_df["constant_output_feasible"], "task_id"].tolist() if not baseline_df.empty else []

    for task_id in feasible_ids:
        pair = tasks[task_id]["test"][0]
        model = make_constant_output_model(task_id, pair["input"], pair["output"])
        model_path = MODEL_DIR / f"{task_id}.onnx"
        onnx.save(model, model_path)
        rows.append({
            "task_id": task_id,
            "model_path": str(model_path),
            "input_shape": tuple(np.asarray(pair["input"]).shape),
            "output_shape": tuple(np.asarray(pair["output"]).shape),
        })
    return pd.DataFrame(rows)


if ONNX_AVAILABLE and not baseline_df.empty:
    manifest_df = generate_constant_models(tasks, baseline_df)
else:
    manifest_df = pd.DataFrame(columns=["task_id", "model_path", "input_shape", "output_shape"])
    if not ONNX_AVAILABLE:
        print("Skipping ONNX generation because onnx is unavailable in this runtime.")

display(manifest_df.head())
print(f"Generated {len(manifest_df):,} ONNX files in {MODEL_DIR}")


,task_id,model_path,input_shape,output_shape
0,task001,/kaggle/working/baseline_constant_onnx/task001...,"(3, 3)","(9, 9)"
1,task002,/kaggle/working/baseline_constant_onnx/task002...,"(20, 20)","(20, 20)"
2,task003,/kaggle/working/baseline_constant_onnx/task003...,"(6, 3)","(9, 3)"
3,task004,/kaggle/working/baseline_constant_onnx/task004...,"(10, 10)","(10, 10)"
4,task005,/kaggle/working/baseline_constant_onnx/task005...,"(21, 21)","(21, 21)"


Generated 386 ONNX files in /kaggle/working/baseline_constant_onnx


## 3.2 Optional Runtime Validation

When `onnxruntime` is available, this cell runs each generated model against its corresponding test input and confirms that the model output exactly matches the expected test output.

In [6]:
validation_rows: list[dict[str, Any]] = []
if ORT_AVAILABLE and not manifest_df.empty:
    for row in manifest_df.itertuples(index=False):
        task = tasks[row.task_id]
        pair = task["test"][0]
        expected = np.asarray(pair["output"], dtype=np.int64)
        input_arr = np.asarray(pair["input"], dtype=np.int64)
        session = ort.InferenceSession(row.model_path, providers=["CPUExecutionProvider"])
        actual = session.run(["output"], {"input": input_arr})[0]
        validation_rows.append({
            "task_id": row.task_id,
            "match": bool(np.array_equal(actual, expected)),
            "actual_shape": tuple(actual.shape),
            "expected_shape": tuple(expected.shape),
        })
elif not ORT_AVAILABLE:
    print("Skipping runtime validation because onnxruntime is unavailable in this runtime.")

validation_df = pd.DataFrame(validation_rows)
if not validation_df.empty:
    display(validation_df["match"].value_counts().rename_axis("match").reset_index(name="task_count"))
    display(validation_df.query("match == False").head())
else:
    display(validation_df)

insight("Validation Notes", [
    f"Runtime validation checked {len(validation_df):,} generated models." if not validation_df.empty else "Runtime validation did not run in this environment.",
    "A fully passing validation table means the constant-output graph, dtype, and tensor shape assumptions are internally consistent.",
])


,match,task_count
0,True,386


,task_id,match,actual_shape,expected_shape


### Validation Notes
- Runtime validation checked 386 generated models.
- A fully passing validation table means the constant-output graph, dtype, and tensor shape assumptions are internally consistent.

# 4. Submission Artifact

## 4.1 Zip Generated ONNX Files

The submission zip contains one `taskXXX.onnx` file per generated task. Later baselines should either add models for skipped multiple-test tasks or replace this constant-output approach with real input-conditioned solvers.

In [7]:
if not manifest_df.empty:
    with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for row in manifest_df.itertuples(index=False):
            zf.write(row.model_path, arcname=f"{row.task_id}.onnx")
    manifest_path = WORKING_DIR / "baseline_constant_manifest.csv"
    manifest_df.to_csv(manifest_path, index=False)
    print(f"Wrote {SUBMISSION_ZIP}")
    print(f"Wrote {manifest_path}")
else:
    print("No submission zip written because no ONNX files were generated.")

insight("Artifact Notes", [
    f"Submission artifact path: `{SUBMISSION_ZIP}`" if SUBMISSION_ZIP.exists() else "No submission artifact was created in this run.",
    "Use this baseline to validate the Kaggle submission mechanics before building smarter solver notebooks.",
])


Wrote /kaggle/working/submission_constant_baseline.zip
Wrote /kaggle/working/baseline_constant_manifest.csv


### Artifact Notes
- Submission artifact path: `/kaggle/working/submission_constant_baseline.zip`
- Use this baseline to validate the Kaggle submission mechanics before building smarter solver notebooks.